In [0]:
import os

# Defining Expected Columns
expected_columns = {
    "olist_customers_dataset": ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"],
    "olist_order_reviews_dataset": ["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp"],
    "olist_orders_dataset": ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"],
    "olist_products_dataset": ["product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    "olist_sellers_dataset": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"],
    "product_category_name_translation": ["product_category_name", "product_category_name_english"]
}

# To handle drift, and decided to skip columns
ignored_columns = {
    "olist_customers_dataset": ["Unwanted_columns"],
    "olist_geolocation_dataset": ["Unwanted_columns"],
    "olist_order_items_dataset": ["Unwanted_columns"],
    "olist_order_payments_dataset": ["Unwanted_columns"],
    "olist_order_reviews_dataset": ["Unwanted_columns"],
    "olist_orders_dataset": ["Unwanted_columns"],
    "olist_products_dataset": ["Unwanted_columns"],
    "olist_sellers_dataset": ["Unwanted_columns"],
    "product_category_name_translation": ["Unwanted_columns"]
}

drifted_tables = []

# Define the path again
raw_path = "abfss://raw@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/"

# Re-generate the 'files' list
# Make sure your authentication/mount is active so this doesn't return 0
all_items = dbutils.fs.ls(raw_path)
files = [f.path for f in all_items if f.name.lower().endswith(".csv")]

if not files:
    print("Warning: No files found. Check your storage connection.")

for file_path in files:
    table_name = os.path.basename(file_path).replace(".csv", "")
    
    # Load the data into 'df'
    df = (spark.read.format("csv")
          .option("header", True)
          .load(file_path))
    
    # Schema Check Logic
    current_bronze_cols = set(df.columns)
    expected_cols = set(expected_columns.get(table_name, []))
    ignored_cols = set(ignored_columns.get(table_name, []))

    new_unhandled_cols = current_bronze_cols - expected_cols - ignored_cols
    missing_cols = expected_cols - current_bronze_cols
    
    if new_unhandled_cols or missing_cols:
        error_detail = f"Table: {table_name} | New: {new_unhandled_cols} | Missing: {missing_cols}"
        drifted_tables.append(error_detail)
        print(f"--> Drift Detected: {error_detail}")
    else:
        print(f"--> Schema Check Passed for {table_name}")

# Decide to run the Pipeline or not
if drifted_tables:
    final_error_report = "\n".join(drifted_tables)
    raise Exception(f"Schema Validation Failed for the following tables:\n{final_error_report}")
else:
    print("--> All tables passed validation.\n--> Proceeding to transformation.")


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, trim, avg, min, max,
                                   coalesce, lit, current_timestamp,
                                   percentile_approx, date_trunc)
                                   
# Parameters and Config
dbutils.widgets.removeAll()
dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

if not table_name:
    dbutils.notebook.exit("Skipped: No table_name provided. Pass table_name via the orchestrator.")

target_table = table_name

bronze_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
silver_path = f"{silver_base}{target_table}"

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Source Load
df = spark.read.format("delta").load(f"{bronze_base}{table_name}")

# Transformation logic
def transform_silver(df, table):
    if table == "olist_customers_dataset":
        pk = "customer_id"
        df_clean = (df.select(
            F.trim(F.col("customer_id")).alias("customer_id"),
            F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
            "customer_zip_code_prefix",
            F.trim(F.col("customer_city")).alias("customer_city"),
            F.trim(F.col("customer_state")).alias("customer_state"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_geolocation_dataset":
        pk = "geolocation_zip_code_prefix"
        df_clean = ((df.filter((F.col("geolocation_lat").cast("float").between(-34, 6)) &
                              (F.col("geolocation_lng").cast("float").between(-74, -34)))
                    .groupby("geolocation_zip_code_prefix")
                    .agg(
                        F.avg("geolocation_lat").cast("decimal(9,6)").alias("geolocation_lat"),
                        F.avg("geolocation_lng").cast("decimal(9,6)").alias("geolocation_lng"),
                        F.min(F.trim(F.col("geolocation_city"))).alias("geolocation_city"),
                        F.max(F.col("geolocation_state")).alias("geolocation_state")))
                        .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))
        
    elif table == "olist_order_items_dataset":
        pk = ["order_id", "order_item_id", "product_id"]
        df_clean = (df.select(
            F.trim(F.col("order_id")).alias("order_id"),
            F.col("order_item_id").cast("int").alias("order_item_id"),
            F.trim(F.col("product_id")).alias("product_id"),
            F.trim(F.col("seller_id")).alias("seller_id"),
            F.date_trunc("second", col("shipping_limit_date").cast("timestamp")).alias("shipping_limit_date"),
            F.col("price").cast("decimal(10,2)").alias("price"),
            F.col("freight_value").cast("decimal(10,2)").alias("freight_value"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_order_payments_dataset":
        pk = ["order_id", "payment_sequential"]
        df_clean = (df.select(
            F.trim(F.col("order_id")).alias("order_id"),
            F.col("payment_sequential").cast("int").alias("payment_sequential"),
            F.trim(F.col("payment_type")).alias("payment_type"),
            F.when(F.col("payment_installments").cast("int") >= 1, F.col("payment_installments").cast("int"))
            .otherwise(1).alias("payment_installments"),
            F.col("payment_value").cast("decimal(10,2)").alias("payment_value"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_order_reviews_dataset":
        pk = ["review_id", "order_id"]
        df_clean = (df.select(
            F.trim(F.col("review_id")).alias("review_id"),
            F.trim(F.col("order_id")).alias("order_id"),
            F.col("review_score").cast("int").alias("review_score"),
            F.coalesce(F.trim(F.col("review_comment_title")), F.lit("N/A")).alias("review_comment_title"),
            F.coalesce(F.trim(F.col("review_comment_message")), F.lit("N/A")).alias("review_comment_message"),
            F.date_trunc("second", F.col("review_creation_date").cast("timestamp")).alias("review_creation_date"),
            F.date_trunc("second", F.col("review_answer_timestamp").cast("timestamp")).alias("review_answer_timestamp"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_orders_dataset":
        pk = "order_id"
        df_clean = (df.select(
            F.trim(F.col("order_id")).alias("order_id"),
            F.trim(F.col("customer_id")).alias("customer_id"),
            F.trim(F.col("order_status")).alias("order_status"),
            F.date_trunc("second", F.col("order_purchase_timestamp").cast("timestamp")).alias("order_purchase_timestamp"),
            F.date_trunc("second", F.col("order_approved_at").cast("timestamp")).alias("order_approved_at"),
            F.date_trunc("second", F.col("order_delivered_carrier_date").cast("timestamp")).alias("order_delivered_carrier_date"),
            F.date_trunc("second", F.col("order_delivered_customer_date").cast("timestamp")).alias("order_delivered_customer_date"),
            F.date_trunc("second", F.col("order_estimated_delivery_date").cast("timestamp")).alias("order_estimated_delivery_date"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_products_dataset":
        pk = "product_id"
        window_spec = Window.partitionBy("product_category_name")

        df_with_medians = (df.withColumn("m_weight", F.percentile_approx("product_weight_g", 0.5).over(window_spec))
                     .withColumn("m_length", F.percentile_approx("product_length_cm", 0.5).over(window_spec))
                     .withColumn("m_height", F.percentile_approx("product_height_cm", 0.5).over(window_spec))
                     .withColumn("m_width", F.percentile_approx("product_width_cm", 0.5).over(window_spec)))

        df_silver = (df_with_medians.select(
            F.trim(F.col("product_id")).alias("product_id"),
            F.coalesce(F.trim(F.col("product_category_name")), F.lit("outros")).alias("product_category_name"),
            F.col("product_name_lenght").cast("int").alias("product_name_length"),
            F.col("product_description_lenght").cast("int").alias("product_description_length"),
            F.col("product_photos_qty").cast("int").alias("product_photos_qty"),
            F.coalesce(F.col("product_weight_g").cast("int"), F.col("m_weight").cast("int")).alias("product_weight_g"),
            F.coalesce(F.col("product_length_cm").cast("int"), F.col("m_length").cast("int")).alias("product_length_cm"),
            F.coalesce(F.col("product_height_cm").cast("int"), F.col("m_height").cast("int")).alias("product_height_cm"),
            F.coalesce(F.col("product_width_cm").cast("int"), F.col("m_width").cast("int")).alias("product_width_cm")))

        df_clean = (df_silver.withColumn(
            "product_volume_cm3", 
            (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")).cast("int"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "olist_sellers_dataset":
        pk = "seller_id"
        df_clean = (df.select(
            F.trim(F.col("seller_id")).alias("seller_id"),
            "seller_zip_code_prefix",
            F.trim(F.col("seller_city")).alias("seller_city"),
            F.trim(F.col("seller_state")).alias("seller_state"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    elif table == "product_category_name_translation":
        pk = "product_category_name"
        df_clean = (df.select(
            F.trim(F.col("product_category_name")).alias("product_category_name"),
            F.trim(F.col("product_category_name_english")).alias("product_category_name_english"))
            .withColumn("silver_load_at", F.date_trunc("second", F.current_timestamp())))

    return df_clean, pk

df_transformed, primary_key = transform_silver(df, table_name)

pk_list = [primary_key] if isinstance(primary_key, str) else primary_key
df_final = df_transformed.dropDuplicates(pk_list)

if DeltaTable.isDeltaTable(spark, silver_path):
    print(f"--> [MERGE] Upserting into Silver: {table_name}")
    dt_silver = DeltaTable.forPath(spark, silver_path)

    merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in pk_list])

    (dt_silver.alias("target")
     .merge(df_final.alias("source"), merge_condition)
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    print(f"--> [INIT] Creating Silver Table: {table_name}")
    df_final.write.format("delta").mode("overwrite").save(silver_path)

# Audit & Exit
dt_final = DeltaTable.forPath(spark, silver_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")